# Exp0.1.5 — Gradient-Calibrated All-Fixes Regularization

Analysis-only notebook for the finalized 5% / 10% / 20% hidden-gradient strength sweep. Training and Slurm submission are intentionally outside this notebook.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("artifacts/experiment_0_1_5_gradient_calibrated_all_fixes/gradient_calibrated_all_fixes_v1")
if not ROOT.exists():
    ROOT = Path("notebooks") / ROOT

runs = pd.read_csv(ROOT / "runs.csv")
summary = pd.read_csv(ROOT / "summary.csv")
history = pd.read_csv(ROOT / "history_long.csv")
calibrations = pd.read_csv(ROOT / "calibrations.csv")
calibration_summary = pd.read_csv(ROOT / "calibration_summary.csv")
gradient_summary = pd.read_csv(ROOT / "gradient_trajectory_summary.csv")
stopping_summary = pd.read_csv(ROOT / "stopping_summary.csv")
paired = pd.read_csv(ROOT / "paired_vs_frozen_exp01.csv")
paired_summary = pd.read_csv(ROOT / "paired_vs_frozen_exp01_summary.csv")
frozen = pd.read_csv(ROOT / "frozen_exp01_binary_context.csv")
frozen_summary = pd.read_csv(ROOT / "frozen_exp01_summary.csv")
comparison = pd.read_csv(ROOT / "comparison_summary.csv")
manifest = json.loads((ROOT / "manifest.json").read_text(encoding="utf-8"))

print("runs:", runs.shape, "history:", history.shape, "calibrations:", calibrations.shape)
assert len(runs) == 90
assert set(runs["target_grad_ratio"]) == {0.05, 0.10, 0.20}
assert len(frozen) == 30


## Experiment contract


In [ ]:
print(json.dumps(manifest, indent=2))


## 5-seed test performance by target strength


In [ ]:
main_cols = [
    "target_grad_ratio", "architecture", "objective",
    "test_ba_mean", "test_ba_std", "test_ba_count",
    "val_ba_mean", "epochs_trained_mean", "best_epoch_mean", "kappa_mean",
]
main = summary[main_cols].sort_values("test_ba_mean", ascending=False)
display(main.reset_index(drop=True))


## Paired test-BA change versus frozen Exp0.1 no-reg


In [ ]:
delta_cols = [
    "target_grad_ratio", "architecture", "objective",
    "delta_test_ba_vs_frozen_exp01_mean",
    "delta_test_ba_vs_frozen_exp01_std",
    "delta_test_ba_vs_frozen_exp01_count",
]
delta = paired_summary[delta_cols].sort_values(
    "delta_test_ba_vs_frozen_exp01_mean", ascending=False
)
display(delta.reset_index(drop=True))

pivot = delta.pivot_table(
    index=["architecture", "objective"],
    columns="target_grad_ratio",
    values="delta_test_ba_vs_frozen_exp01_mean",
)
display(pivot)


In [ ]:
for objective in sorted(runs["objective"].unique()):
    fig, ax = plt.subplots(figsize=(7, 4))
    part = summary[summary["objective"] == objective]
    for architecture in sorted(part["architecture"].unique()):
        sub = part[part["architecture"] == architecture].sort_values("target_grad_ratio")
        ax.errorbar(
            100 * sub["target_grad_ratio"],
            sub["test_ba_mean"],
            yerr=sub["test_ba_std"],
            marker="o",
            label=architecture,
        )
    ax.set_xlabel("Target hidden-gradient ratio (%)")
    ax.set_ylabel("Test balanced accuracy")
    ax.set_title(f"Exp0.1.5 test BA — {objective}")
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(True, alpha=0.25)
    plt.show()


## Calibration stability and frozen kappa


In [ ]:
cal_cols = [
    "target_grad_ratio", "architecture", "objective",
    "calibration_ratio_median_mean", "calibration_ratio_median_std",
    "kappa_mean", "kappa_std",
    "calibration_cosine_mean_mean", "calibration_cosine_mean_std",
]
display(calibration_summary[cal_cols].sort_values(
    ["architecture", "objective", "target_grad_ratio"]
).reset_index(drop=True))

check = calibrations.pivot_table(
    index=["architecture", "objective", "seed"],
    columns="target_grad_ratio",
    values="calibration_ratio_median",
)
print("max within-pair calibration-ratio spread:", (check.max(axis=1) - check.min(axis=1)).max())


## Actual gradient-ratio drift during training


In [ ]:
ratio_col = "first_batch_effective_reg_to_task_hidden_grad_ratio_mean"
for objective in sorted(gradient_summary["objective"].unique()):
    for architecture in sorted(gradient_summary["architecture"].unique()):
        fig, ax = plt.subplots(figsize=(7, 4))
        part = gradient_summary[
            (gradient_summary["objective"] == objective)
            & (gradient_summary["architecture"] == architecture)
        ]
        for target in sorted(part["target_grad_ratio"].unique()):
            sub = part[part["target_grad_ratio"] == target].sort_values("epoch")
            ax.plot(sub["epoch"], sub[ratio_col], marker="o", label=f"target={100*target:.0f}%")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Effective regularizer / task hidden-gradient ratio")
        ax.set_title(f"Gradient-ratio drift — {architecture} — {objective}")
        ax.legend()
        ax.grid(True, alpha=0.25)
        plt.show()


## Task–regularizer gradient cosine


In [ ]:
cos_col = "first_batch_task_reg_hidden_grad_cosine_mean"
cosine_table = gradient_summary[
    ["target_grad_ratio", "architecture", "objective", "epoch", cos_col]
].sort_values(["architecture", "objective", "target_grad_ratio", "epoch"])
display(cosine_table.reset_index(drop=True))


## Early stopping and checkpoint timing


In [ ]:
stop_cols = [
    "target_grad_ratio", "architecture", "objective",
    "epochs_trained_mean", "epochs_trained_std",
    "best_epoch_mean", "best_epoch_std",
    "early_stopped_int_mean",
]
display(stopping_summary[stop_cols].sort_values(
    ["architecture", "objective", "target_grad_ratio"]
).reset_index(drop=True))


## Firing preservation / dead-neuron behavior


In [ ]:
firing = runs.groupby(
    ["target_grad_ratio", "architecture", "objective"], dropna=False
)[[
    "test_last_hidden_events_per_neuron_second",
    "test_last_hidden_dead_neuron_fraction",
    "test_last_hidden_active_step_fraction",
]].agg(["mean", "std"]).reset_index()
firing.columns = [
    "_".join(str(x) for x in col if str(x)) if isinstance(col, tuple) else str(col)
    for col in firing.columns
]
display(firing.sort_values(
    ["architecture", "objective", "target_grad_ratio"]
).reset_index(drop=True))


## Validation trajectories


In [ ]:
val_curve = history.groupby(
    ["target_grad_ratio", "architecture", "objective", "epoch"], dropna=False
)["val_wholecount_ba"].agg(["mean", "std", "count"]).reset_index()

for objective in sorted(val_curve["objective"].unique()):
    for architecture in sorted(val_curve["architecture"].unique()):
        fig, ax = plt.subplots(figsize=(7, 4))
        part = val_curve[
            (val_curve["objective"] == objective)
            & (val_curve["architecture"] == architecture)
        ]
        for target in sorted(part["target_grad_ratio"].unique()):
            sub = part[part["target_grad_ratio"] == target].sort_values("epoch")
            ax.plot(sub["epoch"], sub["mean"], label=f"target={100*target:.0f}%")
        ax.axvline(50, linestyle="--", linewidth=1, label="min epoch")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Validation WholeCount BA")
        ax.set_ylim(0, 1)
        ax.set_title(f"Validation trajectory — {architecture} — {objective}")
        ax.legend()
        ax.grid(True, alpha=0.25)
        plt.show()


## Compact comparison including frozen Exp0.1


In [ ]:
display(comparison.sort_values(
    ["architecture", "objective", "target_grad_ratio"]
).reset_index(drop=True))
